In [4]:
# Cell 1: imports
!pip install xarray zarr gcsfs cftime nc-time-axis -q
import numpy as np
import pandas as pd
import xarray as xr
import gcsfs

gcs = gcsfs.GCSFileSystem(token='anon')

In [5]:
# Cell 2: reload catalog and filter
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')

query = df[
    (df['variable_id'] == 'tas') &
    (df['table_id'] == 'Amon') &
    (df['experiment_id'].isin(['historical', 'ssp245', 'ssp585'])) &
    (df['member_id'] == 'r1i1p1f1') &
    (df['grid_label'] == 'gr')
]

query_filtered = query[query['source_id'] == 'IPSL-CM6A-LR']
print(query_filtered[['source_id', 'experiment_id', 'zstore']])

           source_id experiment_id  \
29231   IPSL-CM6A-LR    historical   
47969   IPSL-CM6A-LR        ssp245   
279556  IPSL-CM6A-LR        ssp585   

                                                   zstore  
29231   gs://cmip6/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/histor...  
47969   gs://cmip6/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR...  
279556  gs://cmip6/CMIP6/ScenarioMIP/IPSL/IPSL-CM6A-LR...  


In [6]:
import numpy as np
import pandas as pd
import xarray as xr
import gcsfs

gcs = gcsfs.GCSFileSystem(token='anon')

# Reload your query_filtered from earlier (IPSL-CM6A-LR)
# If kernel is fresh, re-run the catalog filter cells first

scenarios = ['ssp245', 'ssp585']
target_years = [2030, 2050, 2070, 2100]
results = []

# Also need historical baseline (1850-1900) per grid cell
hist_row = query_filtered[query_filtered['experiment_id'] == 'historical'].iloc[0]
hist_store = gcs.get_mapper(hist_row['zstore'])
hist_ds = xr.open_zarr(hist_store, consolidated=True)

# Compute per-cell baseline mean (1850-1900)
baseline_ds = hist_ds['tas'].sel(time=slice('1850', '1900')).mean(dim='time')
print("Baseline computed")

for scenario in scenarios:
    print(f"Processing {scenario}...")
    row = query_filtered[query_filtered['experiment_id'] == scenario].iloc[0]
    store = gcs.get_mapper(row['zstore'])
    ds = xr.open_zarr(store, consolidated=True)

    for year in target_years:
        print(f"  Year {year}...")
        year_mean = ds['tas'].sel(
            time=slice(str(year), str(year))
        ).mean(dim='time')

        # Anomaly vs per-cell baseline
        anomaly = year_mean - baseline_ds

        # Coarsen to 5-degree grid to keep file small
        coarse = anomaly.coarsen(lat=4, lon=4, boundary='trim').mean()

        df = coarse.to_dataframe(name='tas_anomaly').reset_index()
        df = df[['lat', 'lon', 'tas_anomaly']].dropna()
        df['year'] = year
        df['scenario'] = scenario
        results.append(df)

final = pd.concat(results).reset_index(drop=True)
final['tas_anomaly'] = final['tas_anomaly'].round(3)
print(f"Shape: {final.shape}")
print(final.head())
final.to_csv('spatial_anomaly.csv', index=False)
print("Done! Saved spatial_anomaly.csv")

I0511 23:19:33.490232 3729400 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0511 23:19:33.499078 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499145 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499149 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499151 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499161 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499168 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499170 3729415 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(88, generation: 1)
I0511 23:19:33.499172 3729415 ev_poll_posix.cc:593] FD from fork parent still in p

Baseline computed
Processing ssp245...
  Year 2030...
  Year 2050...
  Year 2070...
  Year 2100...
Processing ssp585...
  Year 2030...
  Year 2050...
  Year 2070...
  Year 2100...
Shape: (10080, 5)
         lat    lon  tas_anomaly  year scenario
0 -88.098587   3.75        1.368  2030   ssp245
1 -88.098587  13.75        1.337  2030   ssp245
2 -88.098587  23.75        1.319  2030   ssp245
3 -88.098587  33.75        1.339  2030   ssp245
4 -88.098587  43.75        1.332  2030   ssp245
Done! Saved spatial_anomaly.csv
